# Original Colab feature and depth experiment
Preserved from the upstream Python export. Google Drive paths, uploads and experimental alignment need configuration. For local video feature extraction, use the root Python CLI. This notebook has not been executed during maintenance.

In [ ]:
# -*- coding: utf-8 -*-
"""Local Features Extraction for Microrobot  Depth Estimation .ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/11gftW5LEwCIQ2V1vkdZKiNF3e19UY7o8

Local Features Extraction on Cropped Frames (Fixed Robots)
"""

!pip install PyWavelets
import cv2
import numpy as np
import random
import matplotlib.pyplot as plt
import pandas as pd
import pywt
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Function to process frames and apply contour detection
def process_frame(frame):
    gray_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    blurred_frame = cv2.GaussianBlur(gray_frame, (5, 5), 0)
    adaptive_thresh = cv2.adaptiveThreshold(blurred_frame, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                            cv2.THRESH_BINARY_INV, 11, 2)
    edges = cv2.Canny(adaptive_thresh, 50, 150)
    kernel = np.ones((5, 5), np.uint8)
    closing = cv2.morphologyEx(edges, cv2.MORPH_CLOSE, kernel)
    contours, _ = cv2.findContours(closing, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)
    return contours, gray_frame

# Function to process frames for focus measures
def process_gray_frame(frame):
    gray_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    blurred_frame = cv2.GaussianBlur(gray_frame, (5, 5), 0)
    resized_frame = cv2.resize(blurred_frame, (256, 256))
    return resized_frame

# Define focus measure functions
def laplacian_of_gaussian(image):
    return cv2.Laplacian(image, cv2.CV_64F).var()

def tenengrad(image):
    sobel_x = cv2.Sobel(image, cv2.CV_64F, 1, 0, ksize=3)
    sobel_y = cv2.Sobel(image, cv2.CV_64F, 0, 1, ksize=3)
    gradient_magnitude = np.sqrt(sobel_x**2 + sobel_y**2)
    return np.mean(gradient_magnitude)

def brenner(image):
    shifted_image = np.roll(image, -2, axis=1)
    return np.sum((image[:, :-2] - shifted_image[:, :-2])**2)

def gray_level_variance(image):
    return np.var(image)

def entropy(image):
    histogram = cv2.calcHist([image], [0], None, [256], [0, 256])
    histogram = histogram / histogram.sum()
    entropy_value = -np.sum(histogram * np.log2(histogram + 1e-10))
    return entropy_value

def wavelet_focus_measure(image):
    coeffs = pywt.wavedec2(image, 'db1', level=4)
    return sum(np.sum(np.abs(coeff)) for coeff in coeffs[1:])

def max_absolute_gradient(image):
    sobel_x = cv2.Sobel(image, cv2.CV_64F, 1, 0, ksize=3)
    sobel_y = cv2.Sobel(image, cv2.CV_64F, 0, 1, ksize=3)
    return np.max(np.abs(sobel_x) + np.abs(sobel_y))

focus_functions = [
    laplacian_of_gaussian,
    tenengrad, brenner,
    gray_level_variance,
    entropy, wavelet_focus_measure,
    max_absolute_gradient
]

# Segment image into smaller grids
def segment_image(image, grid_size):
    segments = []
    h, w = image.shape
    step_h, step_w = h // grid_size[0], w // grid_size[1]
    for i in range(grid_size[0]):
        for j in range(grid_size[1]):
            segment = image[i*step_h:(i+1)*step_h, j*step_w:(j+1)*step_w]
            segments.append(segment)
    return segments

# Extract local features from segmented image
def extract_local_features(image, grid_size, focus_functions):
    segments = segment_image(image, grid_size)
    features = []
    for segment in segments:
        features.extend([func(segment) for func in focus_functions])
    return features

# Function to process video and extract features
def process_video(video_path, focus_functions, bbox_width=300, bbox_height=300, grid_size=(6, 6)):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"Error: Could not open video at {video_path}.")
        return None, None

    processed_frames = []
    focus_features = []

    frame_count = 0  # Initialise frame count for debugging

    while True:
        ret, frame = cap.read()
        if not ret:
            print("End of video or error reading frame.")
            break

        frame_count += 1  # Increment frame count
        print(f"Processing frame {frame_count}")  # Debug print

        contours, gray_frame = process_frame(frame)

        max_area = 0
        main_contour = None
        frame_area = frame.shape[0] * frame.shape[1]

        for contour in contours:
            area = cv2.contourArea(contour)
            if 0.01 * frame_area < area < 0.9 * frame_area:
                if area > max_area:
                    max_area = area
                    main_contour = contour

        if main_contour is not None:
            x, y, w, h = cv2.boundingRect(main_contour)
            center_x, center_y = x + w // 2, y + h // 2
            start_x = max(center_x - bbox_width // 2, 0)
            start_y = max(center_y - bbox_height // 2, 0)
            end_x = min(center_x + bbox_width // 2, frame.shape[1])
            end_y = min(center_y + bbox_height // 2, frame.shape[0])
            cropped_frame = frame[start_y:end_y, start_x:end_x]
            cropped_frame = cv2.resize(cropped_frame, (350, 350))

            gray_cropped_frame = process_gray_frame(cropped_frame)
            processed_frames.append(gray_cropped_frame)

            features = extract_local_features(gray_cropped_frame, grid_size, focus_functions)
            focus_features.append(features)

    cap.release()

    return processed_frames, focus_features

# Define video paths
video_paths = [
    '/content/drive/MyDrive/videos/10_31_13_test.avi',
    '/content/drive/MyDrive/videos/13_45_20_test.avi',
    '/content/drive/MyDrive/videos/10_45_28_test.avi',
    '/content/drive/MyDrive/videos/23_24_20_test.avi',
    '/content/drive/MyDrive/videos/23_27_43_test.avi'
]

# Process videos and save results
for i, video_path in enumerate(video_paths):
    print(f"Processing video {i + 1} at {video_path}")
    processed_frames, focus_features = process_video(video_path, focus_functions)

    # Save local feature data to CSV
    if focus_features:
        df_focus_features = pd.DataFrame(focus_features)
        csv_path = f'/content/drive/MyDrive/df_local_focus_measures_{i + 1}.csv'
        df_focus_features.to_csv(csv_path, index=False)
        print(f"Saved features for video {i + 1} to {csv_path}")

    # Display a random processed frame
    if processed_frames:
        random_frame = random.choice(processed_frames)
        plt.imshow(random_frame, cmap='gray')
        plt.title(f'Random Processed Frame from Video {i + 1}')
        plt.axis('off')
        plt.show()
    else:
        print(f"No frames were processed in video {i + 1}.")

## Visualising data step
df_focus_features = pd.read_csv('/content/drive/MyDrive/df_local_focus_measures_2.csv')

# Display the first few rows of the DataFrame
print("First few rows of df_local_focus_measures_2.csv:")
print(df_focus_features.head())

# Display the shape of the DataFrame
print("\nShape of the DataFrame:")
print(df_focus_features.shape)

## Load Depth Data File

from google.colab import files
import io
import os
import pandas as pd
import matplotlib.pyplot as plt

# Upload depth files from computer
uploaded = files.upload()

# Process each uploaded file
for index, (file_name, file_content) in enumerate(uploaded.items(), start=1):
    df = pd.read_csv(io.StringIO(file_content.decode('utf-8')), header=None, delim_whitespace=True, names=['Time', 'Value'])

    # Ensure time column is in datetime format
    df['Time'] = pd.to_datetime(df['Time'], format='%H:%M:%S.%f')

    # Extract the 'Value' column and drop NaN values
    df_values = df['Value'].dropna()

    # Save values to a CSV
    output_file_path = f'/content/drive/MyDrive/df_values.csv'
    df_values.to_csv(output_file_path, index=False, header=False)

    # Plot values and print number of data points
    plt.figure(figsize=(10, 5))
    plt.plot(df_values, marker='o', linestyle='-')
    plt.title(f'Plot of Extracted Values from {file_name}')
    plt.xlabel('Index')
    plt.ylabel('Depth (μm)')
    plt.grid(True)
    plt.show()

    num_data_points = len(df_values)
    print(f"Number of data points in {file_name}: {num_data_points}")

import pandas as pd

# Load Depth and Focus measures data
df_depth_values = pd.read_csv('/content/drive/MyDrive/df_values.csv', header=None, names=['Depth'])
df_focus_features = pd.read_csv('/content/drive/MyDrive/df_local_focus_measures.csv')

# Ensure the number of focus measures matches the number of depth values
if len(df_focus_features) != len(df_depth_values):
    raise ValueError("The number of focus measures rows must match the number of depth values")

# Combine focus measures with depth data and save to a new CSV file
combined_csv_path = '/content/drive/MyDrive/local_combined_focus_depth_data.csv'
df_combined = pd.concat([df_focus_features, df_depth_values], axis=1)
df_combined.to_csv(combined_csv_path, index=False)
print(f"Combined focus measures and depth data saved to {combined_csv_path}")

# Load and display the combined data
df_combined = pd.read_csv(combined_csv_path)
print("Combined DataFrame:")
print(df_combined.head())

# Define target and features
target = df_combined['Depth']
features = df_combined.drop(columns=['Depth'])

# Display shapes and feature columns
print("Shape of target:", target.shape)
print("Shape of features:", features.shape)
print("Feature columns:")
print(features.columns)

# Perform optimisation
!pip install scikit-optimize

from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor
from sklearn.metrics import r2_score, mean_squared_error
from skopt import BayesSearchCV
import matplotlib.pyplot as plt
import numpy as np

# Define target and features
target = df_combined['Depth']
features = df_combined.drop(columns=['Depth'])

# Define the parameter grid for Bayesian Optimization
param_grid = {
   'n_estimators': (50, 200),             # Range for number of trees
    'max_depth': (3, 10),                  # Range for depth
    'learning_rate': (0.01, 0.2, 'log-uniform'),  # Range for learning rate
    'subsample': (0.6, 0.9),               # Range for subsample ratio
    'colsample_bytree': (0.6, 0.9),        # Range for subsample ratio of columns
    'gamma': (0, 5),                       # Range for minimum loss reduction
    'reg_alpha': (0, 1),                   # Range for L1 regularization term on weights
    'reg_lambda': (1, 5)                   # Range for L2 regularization term on weights
}

# Instantiate XGBRegressor model
xgb = XGBRegressor(random_state=42)

# Perform Bayesian Optimization
bayes_search = BayesSearchCV(estimator=xgb,
                             search_spaces=param_grid,
                             scoring='r2',
                             cv=5,               # 5-fold crossvalidation
                             n_jobs=-1,
                             n_iter=50,          # Number of iterations for the search
                             random_state=42)

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(features, target, test_size=0.2, random_state=42)

# Fit the Bayesian optimization search
bayes_search.fit(X_train, y_train)

# Get best parameters and best model
best_params = bayes_search.best_params_
best_model = bayes_search.best_estimator_

print("Best Parameters:")
print(best_params)

# Train the best model on the training data
best_model.fit(X_train, y_train)

# Evaluate the model on the test data
y_pred = best_model.predict(X_test)
r2 = r2_score(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
print(f"Model R^2 score after hyperparameter tuning: {r2}")
print(f"Mean Squared Error after hyperparameter tuning: {mse}")

## Save model after initial training
import joblib
joblib.dump(best_model, '/content/drive/MyDrive/xgb_model_1.pkl')

#Plot Feature Importance as Heatmaps

import numpy as np

# Extract feature importances from the best model
importances = best_model.feature_importances_

grid_size = (6, 6)
num_focus_functions = 7

# Initialise a list to hold the importance matrices for each focus function
importance_matrices = [np.zeros(grid_size) for _ in range(num_focus_functions)]

# Populate the importance matrices
for i in range(grid_size[0]):
    for j in range(grid_size[1]):
        for k in range(num_focus_functions):
            idx = (i * grid_size[1] + j) * num_focus_functions + k
            importance_matrices[k][i, j] = importances[idx]
import seaborn as sns
import matplotlib.pyplot as plt

# List of focus function names
focus_function_names = [
    "Laplacian of Gaussian",
    "Tenengrad",
    "Brenner",
    "Gray Level Variance",
    "Entropy",
    "Wavelet Focus Measure",
    "Max Absolute Gradient"
]

# Plot the heatmaps for each focus function
for k, function_name in enumerate(focus_function_names):
    plt.figure(figsize=(8, 6))
    sns.heatmap(importance_matrices[k], annot=True, cmap='viridis', cbar_kws={'label': 'Feature Importance'})
    plt.title(f'Feature Importances for {function_name}')
    plt.xlabel('Grid Column')
    plt.ylabel('Grid Row')
    plt.show()

## Visualising data step
df_focus_features_2 = pd.read_csv('/content/drive/MyDrive/df_local_focus_measures_2.csv')

# Display the first few rows of the DataFrame
print("First few rows of df_local_focus_measures_2.csv:")
print(df_focus_features_2.head())

# Display the shape of the DataFrame
print("\nShape of the DataFrame:")
print(df_focus_features_2.shape)

## Load second Depth Data File

from google.colab import files
import io
import os
import pandas as pd
import matplotlib.pyplot as plt

# Upload depth files from the computer
uploaded = files.upload()

# Process each uploaded file
for index, (file_name, file_content) in enumerate(uploaded.items(), start=1):
    # Read the file into a DataFrame
    df = pd.read_csv(io.StringIO(file_content.decode('utf-8')), header=None, delim_whitespace=True, names=['Time', 'Value'])

    # Ensure time column is in datetime format
    df['Time'] = pd.to_datetime(df['Time'], format='%H:%M:%S.%f')

    # Extract the 'Value' column and drop NaN values
    df_values2 = df['Value'].dropna()

    # Save values to a CSV
    output_file_path = f'/content/drive/MyDrive/df_values2.csv'
    df_values2.to_csv(output_file_path, index=False, header=False)

    # Plot values
    plt.figure(figsize=(10, 5))
    plt.plot(df_values2, marker='o', linestyle='-')
    plt.title(f'Plot of Extracted Values from {file_name}')
    plt.xlabel('Index')
    plt.ylabel('Depth (μm)')
    plt.grid(True)
    plt.show()

    # Print number of data points
    num_data_points = len(df_values2)
    print(f"Number of data points in {file_name}: {num_data_points}")

import pandas as pd

# Load Depth and Focus measures data
df_depth_values = pd.read_csv('/content/drive/MyDrive/df_values2.csv', header=None, names=['Depth'])
df_focus_features = pd.read_csv('/content/drive/MyDrive/df_local_focus_measures_2.csv')

# Ensure the number of focus measures matches the number of depth values
if len(df_focus_features) != len(df_depth_values):
    raise ValueError("The number of focus measures rows must match the number of depth values")

# Combine focus measures with depth data and save to a new CSV file
combined_csv_path = '/content/drive/MyDrive/local_combined_focus_depth_data_2.csv'
df_combined = pd.concat([df_focus_features, df_depth_values], axis=1)
df_combined.to_csv(combined_csv_path, index=False)
print(f"Combined focus measures and depth data saved to {combined_csv_path}")

# Load and display the combined data
df_combined = pd.read_csv(combined_csv_path)
print("Combined DataFrame:")
print(df_combined.head())

# Define target and features
target = df_combined['Depth']
features = df_combined.drop(columns=['Depth'])

# Display shapes and feature columns
print("Shape of target:", target.shape)
print("Shape of features:", features.shape)
print("Feature columns:")
print(features.columns)

from xgboost import XGBRegressor
from sklearn.metrics import r2_score, mean_squared_error
import pandas as pd
import joblib

original_data_path = '/content/drive/MyDrive/local_combined_focus_depth_data.csv'

df_original = pd.read_csv(original_data_path)

original_features = df_original.drop(columns=['Depth'])
feature_columns = original_features.columns
new_data_path = '/content/drive/MyDrive/local_combined_focus_depth_data_2.csv'
df_new_data = pd.read_csv(new_data_path)

if not set(feature_columns).issubset(set(df_new_data.columns)):
    raise ValueError("The new data must have the same feature columns as the original training data.")

new_features = df_new_data[feature_columns]
new_target = df_new_data['Depth']

# Load the existing model
model_path = '/content/drive/MyDrive/xgb_model_1.pkl'
best_model = joblib.load(model_path)

# Continue training the model with new data
best_model.fit(new_features, new_target, xgb_model=best_model.get_booster())

# Make predictions on the new data
new_predictions = best_model.predict(new_features)

# Evaluate the model on the new data
new_r2 = r2_score(new_target, new_predictions)
new_mse = mean_squared_error(new_target, new_predictions)
print(f"New Data R^2 score: {new_r2}")
print(f"New Data Mean Squared Error: {new_mse}")

# Save the updated model
updated_model_path = '/content/drive/MyDrive/xgb_model_2.pkl'
joblib.dump(best_model, updated_model_path)
print(f"Updated model saved to {updated_model_path}")

## Visualising data step
df_focus_features_3 = pd.read_csv('/content/drive/MyDrive/df_local_focus_measures_3.csv')

# Display the first few rows of the DataFrame
print("First few rows of df_local_focus_measures_3.csv:")
print(df_focus_features.head())

# Display the shape of the DataFrame
print("\nShape of the DataFrame:")
print(df_focus_features_3.shape)

#Load 3rd Depth Data File
from google.colab import files
import io
import os
import pandas as pd
import matplotlib.pyplot as plt

uploaded = files.upload()
for index, (file_name, file_content) in enumerate(uploaded.items(), start=1):

    df = pd.read_csv(io.StringIO(file_content.decode('utf-8')), header=None, delim_whitespace=True, names=['Time', 'Value'])
    df['Time'] = pd.to_datetime(df['Time'], format='%H:%M:%S.%f')
    df_values3 = df['Value'].dropna()
    output_file_path = f'/content/drive/MyDrive/df_values3.csv'
    df_values3.to_csv(output_file_path, index=False, header=False)

    # Plot values
    plt.figure(figsize=(10, 5))
    plt.plot(df_values3, marker='o', linestyle='-')
    plt.title(f'Plot of Extracted Values from {file_name}')
    plt.xlabel('Index')
    plt.ylabel('Depth (μm)')
    plt.grid(True)
    plt.show()

    # Print number of data points
    num_data_points = len(df_values3)
    print(f"Number of data points in {file_name}: {num_data_points}")

# Load Depth and Focus measures data
df_depth_values = pd.read_csv('/content/drive/MyDrive/df_values3.csv', header=None, names=['Depth'])
df_focus_features = pd.read_csv('/content/drive/MyDrive/df_local_focus_measures_3.csv')

# Ensure the number of focus measures matches the number of depth values
if len(df_focus_features) != len(df_depth_values):
    raise ValueError("The number of focus measures rows must match the number of depth values")

# Combine focus measures with depth data and save to a new CSV file
combined_csv_path = '/content/drive/MyDrive/local_combined_focus_depth_data_3.csv'
df_combined = pd.concat([df_focus_features, df_depth_values], axis=1)
df_combined.to_csv(combined_csv_path, index=False)
print(f"Combined focus measures and depth data saved to {combined_csv_path}")

# Load and display the combined data
df_combined = pd.read_csv(combined_csv_path)
print("Combined DataFrame:")
print(df_combined.head())

# Define target and features
target = df_combined['Depth']
features = df_combined.drop(columns=['Depth'])

from xgboost import XGBRegressor
from sklearn.metrics import r2_score, mean_squared_error
import pandas as pd
import joblib

# Path to the original data file
original_data_path = '/content/drive/MyDrive/local_combined_focus_depth_data.csv'

df_original = pd.read_csv(original_data_path)
original_features = df_original.drop(columns=['Depth'])
feature_columns = original_features.columns

new_data_path = '/content/drive/MyDrive/local_combined_focus_depth_data_3.csv'

df_new_data = pd.read_csv(new_data_path)

if not set(feature_columns).issubset(set(df_new_data.columns)):
    raise ValueError("The new data must have the same feature columns as the original training data.")

new_features = df_new_data[feature_columns]
new_target = df_new_data['Depth']

# Load the existing model
model_path = '/content/drive/MyDrive/xgb_model_2.pkl'
best_model = joblib.load(model_path)

# Continue training the model with new data
best_model.fit(new_features, new_target, xgb_model=best_model.get_booster())

# Make predictions on the new data
new_predictions = best_model.predict(new_features)

# Evaluate the model on the new data
new_r2 = r2_score(new_target, new_predictions)
new_mse = mean_squared_error(new_target, new_predictions)
print(f"New Data R^2 score: {new_r2}")
print(f"New Data Mean Squared Error: {new_mse}")

# Save the updated model
updated_model2_path = '/content/drive/MyDrive/xgb_model_3.pkl'
joblib.dump(best_model, updated_model2_path)
print(f"Updated model saved to {updated_model2_path}")

## Load  Depth Data File
from google.colab import files
import io
import os
import pandas as pd
import matplotlib.pyplot as plt
# Upload depth files from the computer
uploaded = files.upload()

# Process each uploaded file
for index, (file_name, file_content) in enumerate(uploaded.items(), start=1):
    # Read the file into a DataFrame
    df = pd.read_csv(io.StringIO(file_content.decode('utf-8')), header=None, delim_whitespace=True, names=['Time', 'Value'])

    # Ensure time column is in datetime format
    df['Time'] = pd.to_datetime(df['Time'], format='%H:%M:%S.%f')

    # Extract the 'Value' column and drop NaN values
    df_values4 = df['Value'].dropna()

    # Save values to a CSV
    output_file_path = f'/content/drive/MyDrive/df_values4.csv'
    df_values4.to_csv(output_file_path, index=False, header=False)

    # Plot values
    plt.figure(figsize=(10, 5))
    plt.plot(df_values4, marker='o', linestyle='-')
    plt.title(f'Plot of Extracted Values from {file_name}')
    plt.xlabel('Index')
    plt.ylabel('Depth (μm)')
    plt.grid(True)
    plt.show()

    # Print number of data points
    num_data_points = len(df_values4)
    print(f"Number of data points in {file_name}: {num_data_points}")

# Load Depth and Focus measures data
df_depth_values = pd.read_csv('/content/drive/MyDrive/df_values4.csv', header=None, names=['Depth'])
df_focus_features = pd.read_csv('/content/drive/MyDrive/df_local_focus_measures_4.csv')

# Ensure the number of focus measures matches the number of depth values
if len(df_focus_features) != len(df_depth_values):
    raise ValueError("The number of focus measures rows must match the number of depth values")

# Combine focus measures with depth data and save to a new CSV file
combined_csv_path = '/content/drive/MyDrive/local_combined_focus_depth_data_4.csv'
df_combined = pd.concat([df_focus_features, df_depth_values], axis=1)
df_combined.to_csv(combined_csv_path, index=False)
print(f"Combined focus measures and depth data saved to {combined_csv_path}")

# Load and display the combined data
df_combined = pd.read_csv(combined_csv_path)
print("Combined DataFrame:")
print(df_combined.head())

# Define target and features
target = df_combined['Depth']
features = df_combined.drop(columns=['Depth'])

# Display shapes and feature columns
print("Shape of target:", target.shape)
print("Shape of features:", features.shape)
print("Feature columns:")
print(features.columns)

from xgboost import XGBRegressor
from sklearn.metrics import r2_score, mean_squared_error
import pandas as pd
import joblib

# Path to the original data file
original_data_path = '/content/drive/MyDrive/local_combined_focus_depth_data.csv'

# Load the original data to get feature columns
df_original = pd.read_csv(original_data_path)

# Extract features and target from original data
original_features = df_original.drop(columns=['Depth'])
feature_columns = original_features.columns

# Path to the new data file
new_data_path = '/content/drive/MyDrive/local_combined_focus_depth_data_4.csv'

# Load the new data
df_new_data = pd.read_csv(new_data_path)

# Ensure the new data matches the original feature set
if not set(feature_columns).issubset(set(df_new_data.columns)):
    raise ValueError("The new data must have the same feature columns as the original training data.")

# Extract features and target from new data
new_features = df_new_data[feature_columns]
new_target = df_new_data['Depth']

# Load the existing model
model_path = '/content/drive/MyDrive/simpler_xgb_model_3.pkl'
best_model = joblib.load(model_path)

# Continue training the model with new data
best_model.fit(new_features, new_target, xgb_model=best_model.get_booster())

# Make predictions on the new data
new_predictions = best_model.predict(new_features)

# Evaluate the model on the new data
new_r2 = r2_score(new_target, new_predictions)
new_mse = mean_squared_error(new_target, new_predictions)
print(f"New Data R^2 score: {new_r2}")
print(f"New Data Mean Squared Error: {new_mse}")

# Save the updated model
updated_model3_path = '/content/drive/MyDrive/simpler_xgb_model_4.pkl'
joblib.dump(best_model, updated_model3_path)
print(f"Updated model saved to {updated_model3_path}")

## Load Depth Data File
from google.colab import files
import io
import os
import pandas as pd
import matplotlib.pyplot as plt
# Upload depth files from the computer
uploaded = files.upload()

# Process each uploaded file
for index, (file_name, file_content) in enumerate(uploaded.items(), start=1):
    # Read the file into a DataFrame
    df = pd.read_csv(io.StringIO(file_content.decode('utf-8')), header=None, delim_whitespace=True, names=['Time', 'Value'])

    # Ensure time column is in datetime format
    df['Time'] = pd.to_datetime(df['Time'], format='%H:%M:%S.%f')

    # Extract the 'Value' column and drop NaN values
    df_values4 = df['Value'].dropna()

    # Save values to a CSV
    output_file_path = f'/content/drive/MyDrive/df_values5.csv'
    df_values4.to_csv(output_file_path, index=False, header=False)

    # Plot values
    plt.figure(figsize=(10, 5))
    plt.plot(df_values4, marker='o', linestyle='-')
    plt.title(f'Plot of Extracted Values from {file_name}')
    plt.xlabel('Index')
    plt.ylabel('Depth (μm)')
    plt.grid(True)
    plt.show()

    # Print number of data points
    num_data_points = len(df_values4)
    print(f"Number of data points in {file_name}: {num_data_points}")

import pandas as pd
import joblib
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, r2_score
import os

# Paths to the files
features_path = '/content/drive/MyDrive/df_local_focus_measures_5.csv'
target_path = '/content/drive/MyDrive/df_values5.csv'
model_path = '/content/drive/MyDrive/simpler_xgb_model_4.pkl'

# Load the trained model
if os.path.isfile(model_path):
    best_model = joblib.load(model_path)
    print("Model loaded successfully.")
else:
    print(f"Model file does not exist: {model_path}")
    exit()

# Load the features data
if os.path.isfile(features_path):
    df_features = pd.read_csv(features_path)
    print("Features data loaded successfully.")
else:
    print(f"Features data file does not exist: {features_path}")
    exit()

# Load the target data
if os.path.isfile(target_path):
    df_target = pd.read_csv(target_path, header=None)
    print("Target data loaded successfully.")
else:
    print(f"Target data file does not exist: {target_path}")
    exit()

# Extract the 'Value' column and drop NaN values
actual_depth = df_target[0].dropna().values

# Make predictions using the loaded model
predictions = best_model.predict(df_features)

# Calculate MSE and R² score
mse = mean_squared_error(actual_depth, predictions)
r2 = r2_score(actual_depth, predictions)
print(f"Mean Squared Error: {mse}")
print(f"R² Score: {r2}")

# Normalize the actual depth and predictions
scaler = MinMaxScaler()

# Reshape data to fit the scaler
actual_depth_normalized = scaler.fit_transform(actual_depth.reshape(-1, 1)).flatten()
predictions_normalized = scaler.transform(predictions.reshape(-1, 1)).flatten()

# Plot normalized predictions vs. actual depth data with MSE and R² score
plt.figure(figsize=(12, 6))
plt.plot(actual_depth_normalized, label='Actual Depth (Normalized)', color='blue', marker='o')
plt.plot(predictions_normalized, label='Predictions (Normalized)', color='red', linestyle='--')
plt.title('Local features: Normalized Predictions vs. Actual Depth Data')
plt.xlabel('Sample Index')
plt.ylabel('Normalized Depth')
plt.legend()
plt.figtext(0.15, 0.85, f'MSE: {mse:.4f}', fontsize=12, color='black')
plt.figtext(0.15, 0.80, f'R²: {r2:.4f}', fontsize=12, color='black')
plt.tight_layout()
plt.show()